# D5 – Mini-projet : une série temporelle

**Python numérique – devoir n° 5 – à rendre pour le dimanche 18 octobre 2026, 23 h 59** (la séance 6 est le mardi 20)

Bienvenue dans votre cinquième devoir ! Nous allons y mettre en œuvre les notions vues en séance 5 : `to_datetime`, `DatetimeIndex`, la sélection par dates, `resample`, `rolling`, `df.plot` et les figures à plusieurs sous-graphiques avec `matplotlib`. C'est aussi un mini-projet : la dernière partie est ouverte, et c'est vous qui choisissez la question à laquelle répondre.

Durée indicative : 1 h 30.

````{admonition} Consignes de rendu
:class: important

* Ce notebook reste dans le dossier `D5/` de votre dépôt `pe-homework` (celui que le zip du sujet a créé en se dézippant), avec ses figures à côté, puis il doit être **commité et poussé** (`git add`, `git commit`, `git push`) avant la deadline. Attention, c'est la **date du commit** qui fait foi, pas celle du push ; le nom du fichier, lui, n'a pas d'importance (`sujet.ipynb` convient très bien). Le dossier `data/` livré avec le sujet reste lui aussi dans `D5/` : `git add D5` embarque tout le dossier, et c'est très bien ainsi (il ne pèse que quelques kilo-octets).
* Juste avant de rendre, **pensez à ré-exécuter toutes les cellules** via *Kernel → Restart Kernel and Run All Cells* : toutes doivent s'exécuter dans l'ordre et sans erreur. En effet, un notebook qui plante à mi-chemin sera corrigé jusqu'à l'erreur…
* Le notebook lit les données dans `data/comptage-velos.csv` (chemin relatif) ; la figure `serie.png` est quant à elle écrite dans le dossier du notebook.
* Les cellules `# votre code` sont à compléter ; vous pouvez ajouter autant de cellules que vous le souhaitez. Les cellules de test (`assert`) sont là pour vous aider : si elles passent sans erreur, c'est bon signe. Ne les modifiez pas !
* Répondez aux questions "en français" dans une cellule *Markdown*, en deux ou trois phrases. Dans ce devoir, la rédaction compte autant que le code.
````

````{admonition} Ce qui est autorisé
:class: note

* La documentation de `pandas` / `matplotlib`, les notebooks du cours, vos notes.
* Les assistants IA sont tolérés, avec la règle habituelle : **vous devez être capable d'expliquer chaque ligne que vous rendez**, et je pourrai vous demander de le faire à l'oral en séance. Le devoir est individuel.
* En revanche, pas de boucle `for` sur les lignes de la série : tout se fait avec les opérations `pandas`.
````

````{admonition} Les données
:class: warning

`data/comptage-velos.csv` contient le nombre de vélos comptés chaque jour par un compteur automatique installé sur une piste cyclable parisienne, du 1er janvier 2023 au 31 décembre 2024 : une colonne `date` (texte `AAAA-MM-JJ`) et une colonne `comptage`, séparées par `;`.

Attention, **ces données sont simulées** : je les ai produites avec le script `data/generate.py`, en fixant la graine afin que tout le monde travaille sur les mêmes valeurs. Vous pouvez le lire, mais seulement *après* avoir fait le devoir ! Elles ont cependant l'allure de vraies données de comptage, défauts compris : le fichier n'est pas trié, certaines lignes sont en double, certains jours n'ont pas de mesure, et le capteur a eu au moins deux ratés. Ainsi, comme dans la vraie vie, vous allez d'abord nettoyer, puis analyser.
````

## Barème

| Exercice | Contenu | Points |
|---|---|---|
| 1 | Chargement, index temporel, diagnostic et nettoyage | 6 |
| 2 | Rééchantillonnage (`resample`) et lissage (`rolling`) | 4 |
| 3 | Profils hebdomadaire et mensuel, 2023 vs 2024 | 4 |
| 4 | Figure à deux sous-graphiques `serie.png` | 3 |
| 5 | Question ouverte : une question, une figure, trois lignes de conclusion | 3 |
| **Total** | | **20** |
| Bonus | Jour le plus creux, croissance annuelle | +2 |

*Inspiré des exercices de <https://numerique-exos.info-mines.paris>.*

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Exercice 1 – Charger, indexer, nettoyer (6 pts)

**1.a** Lisez le fichier dans une `DataFrame` `df` et affichez `df.info()`. Quel est le type de la colonne `date` ? Pourquoi n'est-ce pas satisfaisant ? Le fichier est-il trié par date ?

In [6]:
df = pd.read_csv("data/comptage-velos.csv", sep=";")
df.info()
print(df["date"].is_monotonic_increasing)

<class 'pandas.DataFrame'>
RangeIndex: 734 entries, 0 to 733
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      734 non-null    str    
 1   comptage  726 non-null    float64
dtypes: float64(1), str(1)
memory usage: 11.6 KB
False


le type de la colonne date est en str ce qui ne permet pas d'utiliser les outils de comparaison de pandas.

In [7]:
assert df.shape == (734, 2)
assert list(df.columns) == ["date", "comptage"]
print("ok")

ok


**Réponse 1.a.** La colonne `date` est initialement du texte (`object`) et non une date pandas. Il faut la convertir pour sélectionner des périodes et réaliser des calculs temporels. Le diagnostic imprimé indique si le fichier est trié ; d'après l'énoncé, il ne l'est pas.

**1.b** Construisez maintenant la `Series` `brut` : les comptages indexés par la date, avec un vrai `DatetimeIndex`, triée par date et en flottants.

In [11]:
brut = pd.Series( df["comptage"].astype(float).to_numpy(), index=pd.to_datetime(df["date"], format="%Y-%m-%d"), name="comptage").sort_index()
brut.head()

date
2023-01-01     304.0
2023-01-02    1658.0
2023-01-03    1450.0
2023-01-04    1902.0
2023-01-05    1715.0
Name: comptage, dtype: float64

In [ ]:
assert isinstance(brut, pd.Series)
assert isinstance(brut.index, pd.DatetimeIndex)
assert brut.index.is_monotonic_increasing
assert len(brut) == 734
assert brut.index[0] == pd.Timestamp("2023-01-01") and brut.index[-1] == pd.Timestamp("2024-12-31")
print("ok")

````{admonition} Pas encore vu en cours : duplicated()
:class: note

`serie.duplicated()` — ou `index.duplicated()` — renvoie un tableau de booléens de même longueur, avec `True` sur les valeurs *déjà rencontrées plus haut* : la première occurrence vaut `False`, les suivantes `True`. Ainsi, sur `[3, 1, 3, 3]`, on obtient `[False, False, True, True]`. Combiné à `~`, il permet donc de ne garder que la première occurrence de chaque valeur.
````

**1.c** Place au diagnostic. Répondez d'abord par du code, puis résumez vos observations dans une cellule Markdown :

* combien de dates apparaissent en double ? lesquelles ?
* combien de valeurs manquantes ? quels jours ?
* observez `brut.describe()` : le minimum et le maximum vous semblent-ils plausibles pour un compteur vélo ? Affichez les jours dont le comptage est nul, et ceux dont le comptage dépasse 3 fois la médiane.

In [ ]:
doublons = brut.index[brut.index.duplicated()]
doublons = brut.index[brut.index.duplicated()].unique()
print(len(doublons), doublons.tolist())
manquants = brut[brut.isna()]
print(len(manquants), manquants.index.tolist())
print(brut.describe())
print(brut[brut == 0])
print(brut[brut > 3 * brut.median()])

In [ ]:
assert brut.index.duplicated().sum() == 3
assert brut.isna().sum() == 8
assert (brut == 0).sum() == 1
assert (brut > 3 * brut.median()).sum() == 1
print("ok")

**Réponse 1.c.** On trouve **3 dates répétées**, **8 comptages manquants**, **une valeur nulle** et **une valeur supérieure à trois fois la médiane**. Les dates exactes et les extrêmes sont affichés par la cellule précédente : il faut les relever à l'exécution. Ces deux valeurs extrêmes sont suspectes pour un compteur et doivent être traitées comme des anomalies, sans présumer de leur cause.

````{admonition} Pas encore vu en cours : interpolate()
:class: note

`serie.interpolate()` remplace chaque `NaN` par une valeur déduite de ses voisins non manquants, par interpolation linéaire par défaut : sur `[10, NaN, NaN, 40]`, on obtient `[10, 20, 30, 40]`. C'est l'alternative à `fillna` quand les données ont un ordre naturel, comme ici le temps.
````

**1.d** Passons au nettoyage. Construisez la série propre `velos`, en trois temps :

1. supprimez les doublons d'index en ne gardant que la première occurrence (`duplicated()`) ;
2. remplacez les deux valeurs aberrantes (comptage nul ou supérieur à 3 fois la médiane) par `NaN` ;
3. remplissez tous les `NaN` (les 8 jours vides + les 2 aberrants) par interpolation linéaire.

Vérifiez enfin que `velos` a exactement une valeur par jour du 1er janvier 2023 au 31 décembre 2024 (soit 731 jours), et plus aucun `NaN`.

In [ ]:
velos = brut[~brut.index.duplicated(keep="first")].copy()
mediane = velos.median()
velos[(velos == 0) | (velos > 3 * mediane)] = np.nan
velos = velos.interpolate(method="linear")
print("Nombre de jours :", len(velos))
print("Valeurs manquantes après nettoyage :", velos.isna().sum())

In [ ]:
assert len(velos) == 731
assert velos.isna().sum() == 0
assert not velos.index.duplicated().any()
assert (velos.index == pd.date_range("2023-01-01", "2024-12-31", freq="D")).all()
assert velos.min() > 100 and velos.max() < 6000
assert abs(velos.loc["2024-07-04"] - (velos.loc["2024-07-03"] + velos.loc["2024-07-05"]) / 2) < 1
print("ok")

## Exercice 2 – Rééchantillonner et lisser (4 pts)

**2.a** Calculez :

* `hebdo` : la moyenne par semaine ;
* `mensuel` : la moyenne par mois (l'alias de fréquence des mois est `"ME"`, *month end*, depuis pandas 2.2 ; `"M"` si votre pandas est plus ancien) ;
* `annuel` : le **total** par année.

Combien de lignes ont `hebdo` et `mensuel` ? Pourquoi `hebdo` n'a-t-il pas exactement 731 / 7 lignes ?

In [ ]:
hebdo = velos.resample("W").mean()
mensuel = velos.resample("ME").mean()  # "M" avec pandas ancien
annuel = velos.resample("YE").sum()   # "Y" avec pandas ancien
print("Semaines :", len(hebdo), "| Mois :", len(mensuel))
print("Totaux annuels :")
print(annuel)

In [ ]:
assert len(hebdo) == 106 and len(mensuel) == 24 and len(annuel) == 2
assert abs(mensuel.sum() * 30.4 - velos.sum()) / velos.sum() < 0.05
assert annuel.iloc[1] > annuel.iloc[0]
print("ok")

**Réponse 2.a.** `hebdo` comporte **106** lignes et `mensuel` **24**. Le découpage hebdomadaire dépend des limites des semaines calendaires (par défaut, les semaines se terminent le dimanche) : les semaines aux extrémités peuvent être incomplètes, et 731 jours ne se regroupent donc pas simplement en blocs de sept.

**2.b** Calculez `lisse`, la moyenne glissante **centrée** sur 7 jours. Combien de `NaN` contient-elle, et où ? Pourquoi 7 jours est-il un bon choix de fenêtre pour cette série ? Enfin, tracez `velos` (trait fin, transparent) et `lisse` sur le même graphique, avec la méthode `.plot()` des `Series`.

In [ ]:
lisse = velos.rolling(window=7, center=True).mean()
print("Nombre de NaN dans lisse :", lisse.isna().sum())
print("Dates des NaN :", lisse.index[lisse.isna()].strftime("%Y-%m-%d").tolist())
ax = velos.plot(figsize=(11, 4), linewidth=0.7, alpha=0.3, label="Quotidien")
lisse.plot(ax=ax, linewidth=2, label="Moyenne mobile sur 7 jours")
ax.set(title="Comptage quotidien de vélos et tendance lissée", xlabel="Date", ylabel="Vélos par jour")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
assert len(lisse) == 731
assert lisse.isna().sum() == 6
assert abs(lisse.loc["2023-06-15"] - velos.loc["2023-06-12":"2023-06-18"].mean()) < 1e-9
print("ok")

**Réponse 2.b.** Le lissage centré produit **6 NaN**, pour les trois premiers et les trois derniers jours de la série, car une fenêtre complète de 7 observations n'y existe pas. La fenêtre de 7 jours couvre un cycle hebdomadaire complet, ce qui réduit les oscillations entre jours ouvrés et week-end.

## Exercice 3 – Profils (4 pts)

**3.a** Commençons par le profil hebdomadaire : calculez `profil_semaine`, la moyenne des comptages pour chaque jour de la semaine, avec 0 = lundi … 6 = dimanche. Remplacez ensuite l'index `0..6` par les noms des jours (`"lun"`, `"mar"`, … `"dim"`). De combien (en %) le dimanche est-il plus bas que la moyenne des jours de semaine (lundi à vendredi) ?

In [ ]:
profil_semaine = velos.groupby(velos.index.dayofweek).mean()
profil_semaine.index = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
moyenne_semaine = profil_semaine.loc[["lun", "mar", "mer", "jeu", "ven"]].mean()
baisse_dimanche = 100 * (moyenne_semaine - profil_semaine["dim"]) / moyenne_semaine
print(profil_semaine)
print(f"Le dimanche est inférieur de {baisse_dimanche:.1f} % à la moyenne lundi-vendredi.")

In [ ]:
assert profil_semaine.shape == (7,)
assert list(profil_semaine.index) == ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
assert profil_semaine["sam"] < 0.75 * profil_semaine["lun"]
assert profil_semaine["dim"] < profil_semaine["sam"]
print("ok")

**Réponse 3.a.** Le code calcule précisément le pourcentage de baisse du dimanche par rapport à la moyenne des **cinq jours ouvrés**. Le profil montre un comptage plus faible le week-end ; reportez le pourcentage affiché lors de l'exécution.

````{admonition} Pas encore vu en cours : idxmax et idxmin
:class: note

`serie.idxmax()` renvoie l'**étiquette** de la plus grande valeur (ici un numéro de mois), alors que `serie.max()` renvoie la valeur elle-même ; `idxmin` fait de même pour la plus petite. Sur une série indexée par des dates, on récupère donc directement le `Timestamp` du maximum.
````

**3.b** Intéressons-nous maintenant à la saisonnalité : calculez `profil_mois`, la moyenne pour chaque mois de l'année (1 à 12), les deux années confondues. Quel est le mois le plus chargé ? Et que se passe-t-il en août ?

In [ ]:
profil_mois = velos.groupby(velos.index.month).mean()
print(profil_mois)
print("Mois le plus chargé :", profil_mois.idxmax())
print("Moyenne d'août :", round(profil_mois.loc[8], 1))
print("Juillet / août / septembre :", profil_mois.loc[[7, 8, 9]].round(1).to_dict())

In [ ]:
assert profil_mois.shape == (12,) and list(profil_mois.index) == list(range(1, 13))
assert profil_mois.idxmax() in (6, 7)
assert profil_mois[8] < profil_mois[7] and profil_mois[8] < profil_mois[9]
assert profil_mois.idxmin() in (1, 12)
print("ok")

**Réponse 3.b.** Le mois le plus chargé est identifié par `idxmax()` (selon les tests : **juin ou juillet**). Le mois d'août connaît un creux par rapport à juillet et septembre, compatible avec une baisse des déplacements réguliers pendant les vacances ; ce n'est qu'une interprétation possible.

**3.c** Opposons enfin 2023 à 2024 : avec la sélection par chaîne de date, calculez la moyenne journalière de chaque année, puis la croissance en % de 2023 à 2024 (variable `croissance`). Posez-vous ensuite la même question mois par mois : construisez la table `par_mois` de forme `(12, 2)` (lignes = mois, colonnes = années) — `unstack` peut aider — et repérez le mois où la progression a été la plus forte.

In [ ]:
moyenne_2023 = velos.loc["2023"].mean()
moyenne_2024 = velos.loc["2024"].mean()
croissance = 100 * (moyenne_2024 / moyenne_2023 - 1)
par_mois = velos.groupby([velos.index.month, velos.index.year]).mean().unstack()
par_mois.index.name = "mois"
par_mois.columns.name = "annee"
progression_mensuelle = 100 * (par_mois[2024] / par_mois[2023] - 1)
print(f"Moyenne 2023 : {moyenne_2023:.1f}; moyenne 2024 : {moyenne_2024:.1f}")
print(f"Croissance moyenne journalière : {croissance:.2f} %")
print(par_mois.round(1))
print(f"Plus forte progression : mois {progression_mensuelle.idxmax()} ({progression_mensuelle.max():.1f} %)")

In [ ]:
assert 7 < croissance < 14
assert par_mois.shape == (12, 2)
assert list(par_mois.columns) == [2023, 2024]
assert (par_mois[2024] > par_mois[2023]).sum() >= 10
print("ok")

**Réponse 3.c.** La croissance compare les **moyennes quotidiennes** de 2024 et de 2023. La table `par_mois` permet de comparer les mêmes mois entre les deux années ; la cellule affiche le mois qui a le plus progressé ainsi que son pourcentage. Les chiffres définitifs nécessitent le CSV.

## Exercice 4 – Figure à deux sous-graphiques (3 pts)

Construisez une figure `fig, (ax_haut, ax_bas) = plt.subplots(2, 1, figsize=(11, 8))` :

* en haut : la série journalière `velos` (trait fin, transparent) et la série lissée `lisse` (trait plus épais), avec une légende, un titre et un nom d'axe des ordonnées ;
* en bas : le profil hebdomadaire `profil_semaine` en barres, avec un titre et un nom d'axe des ordonnées.

Terminez par `fig.tight_layout()` et enregistrez dans `serie.png` (`fig.savefig`).

In [ ]:
fig, (ax_haut, ax_bas) = plt.subplots(2, 1, figsize=(11, 8))
velos.plot(ax=ax_haut, linewidth=0.7, alpha=0.3, label="Comptage journalier")
lisse.plot(ax=ax_haut, linewidth=2, label="Moyenne mobile 7 jours")
ax_haut.set_title("Comptage quotidien des vélos (2023–2024)")
ax_haut.set_ylabel("Vélos par jour")
ax_haut.set_xlabel("Date")
ax_haut.legend()
profil_semaine.plot(kind="bar", ax=ax_bas, color="steelblue", label="Moyenne par jour")
ax_bas.set_title("Profil hebdomadaire moyen")
ax_bas.set_ylabel("Vélos par jour")
ax_bas.set_xlabel("Jour de la semaine")
ax_bas.legend()
fig.tight_layout()
fig.savefig("serie.png", dpi=150)
plt.show()

In [ ]:
assert os.path.exists("serie.png")
print("ok")

## Exercice 5 – Question ouverte (3 pts)

C'est ici que vous prenez la main ! Choisissez **une** question sur cette série, puis répondez-y avec une figure (au moins une courbe ou un diagramme, légendée et titrée) et trois lignes de conclusion en Markdown. Voici quelques idées, mais une question à vous est la bienvenue :

* le creux d'août a-t-il la même ampleur en 2023 et en 2024 ?
* l'écart week-end / semaine est-il le même en été et en hiver ?
* quelle semaine de l'année est la plus chargée ? la plus creuse ? est-ce la même les deux années ?
* la croissance entre 2023 et 2024 est-elle uniforme sur l'année, ou concentrée sur certains mois ?
* que se passe-t-il autour des jours fériés que vous connaissez ?

Côté barème, je compte un point pour la pertinence de la méthode, un point pour une figure lisible et correcte, et un point pour une conclusion chiffrée et honnête.

In [ ]:
# Question : La croissance entre 2023 et 2024 est-elle uniforme selon les mois ?
progression_mensuelle = 100 * (par_mois[2024] / par_mois[2023] - 1)
ax = progression_mensuelle.plot(kind="bar", figsize=(10, 4), color="teal", label="Croissance 2024 / 2023")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Évolution du comptage moyen par mois : 2024 vs 2023",
       xlabel="Mois", ylabel="Variation en %")
ax.legend()
plt.tight_layout()
plt.savefig("croissance_mensuelle.png", dpi=150)
plt.show()

# Les valeurs et les trois phrases ci-dessous sont calculées automatiquement.
# Recopiez-les, après exécution, dans la cellule Markdown suivante.
from IPython.display import display, Markdown
mois_max = int(progression_mensuelle.idxmax())
mois_min = int(progression_mensuelle.idxmin())
conclusion = (
    f"1. La moyenne journalière augmente globalement de **{croissance:.1f} %** entre 2023 et 2024.\n"
    f"2. La progression la plus forte concerne le mois **{mois_max}** (**{progression_mensuelle.max():.1f} %**), "
    f"et la plus faible le mois **{mois_min}** (**{progression_mensuelle.min():.1f} %**).\n"
    f"3. La croissance n'est donc **pas uniforme** selon les mois : "
    f"l'écart entre les extrêmes est de **{progression_mensuelle.max() - progression_mensuelle.min():.1f} points**."
)
display(Markdown(conclusion))

### Conclusion de la question ouverte

La cellule ci-dessus affiche automatiquement **trois phrases chiffrées** en Markdown à partir de vos données. Après avoir exécuté le notebook avec `data/comptage-velos.csv`, copiez ces trois phrases ici pour satisfaire exactement à la consigne de rédaction en cellule Markdown.

## Bonus (+2)

**Bo.1** Quel est le jour le plus creux de toute la série ? Et le plus chargé ? Pour chacun, donnez le jour de la semaine (`.day_name()`) et proposez une explication.

````{admonition} Pas encore vu en cours : pct_change
:class: note

`serie.pct_change()` renvoie la variation *relative* d'une valeur à la suivante : sur `[100, 110, 99]`, elle donne `[NaN, 0.10, -0.10]`. La première case vaut `NaN` faute de valeur précédente, et le résultat est une proportion, à multiplier par 100 pour l'exprimer en pourcentage.
````

**Bo.2** Calculez la croissance annuelle en % à partir des totaux annuels `annuel`. Pourquoi ce chiffre est-il légèrement différent de `croissance` (exercice 3.c) ?

In [ ]:
jour_min = velos.idxmin()
jour_max = velos.idxmax()
print(f"Jour le plus creux : {jour_min.date()} ({jour_min.day_name()}) = {velos.min():.0f} vélos")
print(f"Jour le plus chargé : {jour_max.date()} ({jour_max.day_name()}) = {velos.max():.0f} vélos")
print("Interprétations possibles (hypothèses) : météo, vacances, jour férié ou jour ouvré.")
croissance_totaux = 100 * (annuel.pct_change().iloc[1])
print(f"Croissance calculée sur les totaux annuels : {croissance_totaux:.2f} %")
print("2024 a 366 jours, contre 365 en 2023 : les totaux annuels dépendent aussi du nombre de jours.")

**Réponse bonus.** Les jours extrêmes sont identifiés par `idxmin()` et `idxmax()` ; une explication liée à la météo ou aux vacances reste une **hypothèse** sans données complémentaires. La croissance des totaux annuels diffère légèrement de celle des moyennes journalières parce que **2024 est bissextile (366 jours)** tandis que **2023 compte 365 jours**.

Bon courage à tous ! Pour toute question sur le sujet, écrivez-moi à [aurelien.noce@minesparis.psl.eu](mailto:aurelien.noce@minesparis.psl.eu), en mettant [aurelien.noce@imagine-app.fr](mailto:aurelien.noce@imagine-app.fr) en copie : la messagerie de l'école perd parfois des mails, la seconde adresse sert de filet.

Au plaisir de vous lire,

votre responsable de cours, Aurélien Noce